Check pointer is a component that save state of the graph after each node execution within a super-step

The saved information in additional metadata object called StateSnapshot

StateSnapshot enable for example, human-in-the-loop workflows, modify the steps during execution

Threads: collection of related checkpoints(StateSnapshot)
Threads have their special id and provide continuity

Short-term memory: reset when the kernel restarted
Long-term memory: backed by the external storage

In [ ]:
from langgraph.graph import START, END, StateGraph, MessagesState
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, RemoveMessage, BaseMessage
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
class State(MessagesState):
    summary: str

In [ ]:
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="not-needed",
    model="qwen2.5-3b-instruct",
    temperature=0,
    max_completion_tokens=250
)

In [ ]:
def ask_question(state: State) -> State:

    print(f"\n------->  ENTERING ask_question:")

    question = "what is your question?"
    print(question)

    return State(messages= [AIMessage(question), HumanMessage(input())])

In [ ]:
def chatbot(state: State) -> State:  #this function is good when the number of nodes increasing
    
    print(f"\n------->  ENTERING chatbot:")

    system_message = f'''
    Here is a quick summary of what's been discussed so far:
    {state.get("summary", "")}
    
    keep this in mind as you answer the next question
    '''

    response = llm.invoke([SystemMessage(system_message)] + state["messages"])
    response.pretty_print()

    return State(messages= [response])

In [ ]:
def summarize_messages(state: State) -> State:

    print(f"\n------->  ENTERING summarized_messages:")

    new_conversations = ""
    for i in state["messages"]:
        new_conversations += f"{i.type}: {i.content}\n\n"

    summary_instructions = f'''
    update the ongoing summary by incorporating the new lines of conversation below.
    build upon the pervious summary rather than repeating it so that the result reflects the most recent context
    and developments

    pervious summary:
    {state.get("summary", "")}

    new conversation:
    {new_conversations}
    '''
    print(summary_instructions)

    summary = llm.invoke([HumanMessage(summary_instructions)])

    remove_messages = [RemoveMessage(id= i.id) for i in state["messages"][:]]

    return State(messages= remove_messages, summary= summary.content)

In [ ]:
graph = StateGraph(State)

In [ ]:
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("summarize_messages", summarize_messages)

graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "summarize_messages")
graph.add_edge("summarize_messages", END)

Instruct the graph to create In-memory checkpoints and associate them with the specific thread, enabling thread-level persistant

In [ ]:
checkpointer = InMemorySaver()
graph_compiled = graph.compile(checkpointer) 

In [ ]:
config1 = {"configurable": {"thread_id":"1"}}

In [ ]:
graph_compiled.invoke(State(), config= config1)

In [ ]:
graph_state = [i for i in graph_compiled.get_state_history(config1)]

In [ ]:
for i in graph_state:
    print(i)

In [ ]:
for i in graph_state[::-1]:
    print(f'''
Messages: {i.values["messages"]}
Summary: {i.values.get("summary", "")}
Next: {i.next}
Step: {i.metadata["step"]}
''')
